# random_backbone_baseline

**作為 SSL、SL 的 lower bound。**  
Random ResNet-18 → Freeze backbone → 512→10 Linear Probe → Test Accuracy

## step1:Import + Dataset

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet18

import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


# CIFAR-10 normalization
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
])

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

batch_size = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print("Train:", len(train_dataset))
print("Test:", len(test_dataset))

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


c:\Users\user\anaconda3\envs\summer\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train: 50000
Test: 10000


## step2:建立「隨機且永遠不訓練」的 Backbone

In [2]:
backbone = resnet18(weights=None)

# 與前兩個實驗完全相同的 CIFAR-10 修改
backbone.conv1 = nn.Conv2d(
    3,
    64,
    kernel_size=3,
    stride=1,
    padding=1,
    bias=False
)

backbone.maxpool = nn.Identity()

# 移除原本分類層
backbone.fc = nn.Identity()

backbone = backbone.to(device)

# 關鍵：Freeze random backbone
for param in backbone.parameters():
    param.requires_grad = False

backbone.eval()

print(
    "Trainable backbone parameters:",
    sum(p.numel() for p in backbone.parameters() if p.requires_grad)
)

Trainable backbone parameters: 0


## step3:建立 Linear Probe

In [3]:
linear_classifier = nn.Linear(512, 10).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    linear_classifier.parameters(),
    lr=1e-3,
    weight_decay=1e-6
)

print(linear_classifier)

Linear(in_features=512, out_features=10, bias=True)


## step4:正式 Linear Probing 100 Epochs

In [4]:
num_epochs = 100

loss_history = []
test_accuracy_history = []

start_time = time.time()

for epoch in range(1, num_epochs + 1):

    # =====================
    # Train Linear Layer
    # =====================
    linear_classifier.train()
    backbone.eval()

    total_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Random backbone 永遠不更新
        with torch.no_grad():
            features = backbone(images)

        outputs = linear_classifier(features)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    loss_history.append(avg_loss)

    # =====================
    # Test
    # =====================
    linear_classifier.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            features = backbone(images)
            outputs = linear_classifier(features)

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    test_acc = 100.0 * correct / total
    test_accuracy_history.append(test_acc)

    print(
        f"Epoch [{epoch:3d}/{num_epochs}] "
        f"| Loss: {avg_loss:.4f} "
        f"| Test Accuracy: {test_acc:.2f}%"
    )


elapsed = time.time() - start_time

print("\n===== Random Backbone Finished =====")
print(f"Training Time: {elapsed/60:.2f} minutes")
print(f"Final Test Accuracy: {test_accuracy_history[-1]:.2f}%")
print(f"Best Test Accuracy: {max(test_accuracy_history):.2f}%")

Epoch [  1/100] | Loss: 2.2084 | Test Accuracy: 22.22%
Epoch [  2/100] | Loss: 2.0805 | Test Accuracy: 28.89%
Epoch [  3/100] | Loss: 2.0175 | Test Accuracy: 29.80%
Epoch [  4/100] | Loss: 1.9717 | Test Accuracy: 27.78%
Epoch [  5/100] | Loss: 1.9477 | Test Accuracy: 32.35%
Epoch [  6/100] | Loss: 1.9191 | Test Accuracy: 33.39%
Epoch [  7/100] | Loss: 1.9011 | Test Accuracy: 35.03%
Epoch [  8/100] | Loss: 1.8826 | Test Accuracy: 35.02%
Epoch [  9/100] | Loss: 1.8723 | Test Accuracy: 35.34%
Epoch [ 10/100] | Loss: 1.8608 | Test Accuracy: 36.59%
Epoch [ 11/100] | Loss: 1.8422 | Test Accuracy: 35.73%
Epoch [ 12/100] | Loss: 1.8355 | Test Accuracy: 35.85%
Epoch [ 13/100] | Loss: 1.8288 | Test Accuracy: 36.23%
Epoch [ 14/100] | Loss: 1.8180 | Test Accuracy: 37.12%
Epoch [ 15/100] | Loss: 1.8067 | Test Accuracy: 37.65%
Epoch [ 16/100] | Loss: 1.7975 | Test Accuracy: 37.15%
Epoch [ 17/100] | Loss: 1.7921 | Test Accuracy: 38.34%
Epoch [ 18/100] | Loss: 1.7860 | Test Accuracy: 37.63%
Epoch [ 19